In [4]:
import torch
import torch.nn as nn
import re
from collections import Counter
import pickle
import pandas as pd
from torch.nn.utils.rnn import pad_sequence
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
class Vocabulary:
    def __init__(self, freq_threshold=5): 
        self.itos = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold
        
    def __len__(self):
        return len(self.itos)
    
    @staticmethod
    def tokenizer_text(text):
        text = str(text).lower() 
        text = text.replace("n't", " not")
        text = text.replace("'s", " is")
        text = text.replace("'re", " are")
        text = text.replace("-", " ")
        text = re.sub(r'[^a-z\s]', '', text)
        return text.split()
    
    def build_vocabulary(self, sentence_list):
        # Builds the word-to-index mapping
        frequencies = Counter()
        idx = 4 
        
        for sentence in sentence_list:
            for word in self.tokenizer_text(sentence):
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
                    
    def numericalize(self, text):
        tokenized_text = self.tokenizer_text(text)       
        # Converts words to numbers and automatically wraps them in START and END tokens
        return [
            self.stoi["<START>"]
        ] + [
            self.stoi.get(token, self.stoi["<UNK>"]) for token in tokenized_text
        ] + [
            self.stoi["<END>"]
        ]

In [6]:
class CapsCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        # The dataloader will pass a 'batch' which is a list of tuples: (image, caption)
        images = torch.stack([item[0] for item in batch], dim=0)
        # Converting them to PyTorch tensors
        targets = [torch.tensor(item[1]) for item in batch]     
        # Paddng the sequences
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return images, targets

In [7]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=3):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.embed_dropout = nn.Dropout(0.5)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, 
                            batch_first=True, dropout=0.5)      
        self.linear = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, features, captions):
        embeddings = self.embed_dropout(self.embed(captions[:, :-1]))
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)      
        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs

print("LSTM Architecture is ready!")

LSTM Architecture is ready!


# Integration

In [8]:
file_path = 'captions.txt' 

print(f"Attempting to load captions from: {file_path}")

try:
    df = pd.read_csv(file_path)
    all_captions_list = df['caption'].astype(str).tolist()
    print(f"Successfully loaded {len(all_captions_list)} captions.")
    
    # 3. Initialize and build the vocabulary
    print("Building vocabulary")
    vocab = Vocabulary(freq_threshold=5)
    vocab.build_vocabulary(all_captions_list)

    with open("vocab.pkl", "wb") as f:
        pickle.dump(vocab, f)
        
    print(f" Vocabulary built with {len(vocab)} unique words.")
    print("'vocab.pkl' has been generated and saved to the directory.")
except FileNotFoundError:
    print(f" Error: Could not find '{file_path}'.")
except Exception as e:
    print(f" An error occurred while parsing the file: {e}")

Attempting to load captions from: captions.txt
Successfully loaded 40455 captions.
Building vocabulary
 Vocabulary built with 2973 unique words.
'vocab.pkl' has been generated and saved to the directory.
